# Shoaib HAR Feature Importance

Build the pairwise feature-importance knowledge base used by the Shoaib ZARA inference example.


In [ ]:
import os
from collections import Counter
import pickle
import pandas as pd
import random
import numpy as np
from collections import defaultdict
import math
from typing import Dict
from scipy.signal import stft
import math
from scipy.stats import iqr, skew, kurtosis, entropy
from scipy.signal import welch, detrend, correlate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

output_path = "./dataset/shoaib"

with open(os.path.join(output_path, 'shoaib_database_segments.pkl'), 'rb') as f:
    all_database_segments = pickle.load(f)

with open(os.path.join(output_path, f'shoaib_test_data.pkl'), 'rb') as f:
    all_test_segments = pickle.load(f)

with open(os.path.join(output_path, 'shoaib_database_labels.pkl'), 'rb') as f:
    all_database_labels = pickle.load(f)

with open(os.path.join(output_path, f'shoaib_test_labels.pkl'), 'rb') as f:
    all_test_labels = pickle.load(f)

activity_map = {
    0: "walking",
    1: "standing",
    2: "jogging",
    3: "sitting",
    4: "biking",
    5: "upstairs",
    6: "downstairs",
}

label2id = {}
id2label = {}
for i, (key, value) in enumerate(activity_map.items()):
    label2id[value]=i
    id2label[i]=value
print(f"label2id:\n{label2id}")
print(f"id2label:\n{id2label}")

from collections import defaultdict, Counter

# Count activity samples by subject.
activity_subject_counts = defaultdict(Counter)
for lbl in all_test_labels:
    act = lbl["activity"]
    subj = lbl["subject"]
    activity_subject_counts[act][subj] += 1

# Print the subject distribution for each activity.
for act, subj_counts in activity_subject_counts.items():
    print(f"Activity = {act}")
    for subj, cnt in subj_counts.items():
        print(f"    Subject {subj}: {cnt}")
    print()


## Feature Constants

Set signal-processing constants for the Shoaib sampling rate and window size.


In [ ]:
# Constants
FS = 50                      # Hz
DT = 1.0 / FS
SEQ_LEN = 100
DEG2RAD = math.pi / 180.0
ORDER=4


## Signal Feature Functions

Define reusable feature extractors for all five body placements.


In [ ]:
from scipy.integrate import cumulative_trapezoid
from scipy.signal import butter, filtfilt
from spectrum import arburg
from scipy.spatial.distance import pdist, squareform
import pywt
import numpy as np
import math
import itertools
from collections import Counter

def spectral_centroid(freqs, psd):
    """Compute the power-weighted average frequency of a spectrum."""
    return np.sum(freqs * psd) / (np.sum(psd) + 1e-12)

def compute_acf(x: np.ndarray, max_lag: int) -> np.ndarray:
    """Compute a normalized autocorrelation sequence up to a maximum lag."""
    x = x - x.mean()
    N = len(x)
    max_lag = min(max_lag, N-1)

    if np.all(x == 0) or np.var(x) < 1e-8:
        acf = np.zeros(max_lag+1, dtype=float)
        acf[0] = 1.0
        return acf

    r = correlate(x, x, mode='full')
    r = r[N-1 : N-1 + max_lag + 1]

    denom = r[0] if abs(r[0]) > 1e-12 else 1e-12
    acf = r / denom

    acf[0] = 1.0

    return acf

def zero_cross_centered(channel_data):
    """Count zero crossings after removing the signal mean."""
    centered = channel_data - np.mean(channel_data)
    signs = np.sign(centered)
    return np.sum(signs[:-1] * signs[1:] < 0)

def band_power(psd_freq, psd_val, fmin, fmax):
    """Integrate spectral power over a frequency band."""
    idx = np.logical_and(psd_freq >= fmin, psd_freq <= fmax)
    return np.trapz(psd_val[idx], psd_freq[idx])

def lowpass(data, cutoff=0.3, fs=FS, order=3):
    """Keep low-frequency components, which mainly capture gravity."""
    b, a = butter(order, cutoff/(fs/2), btype='low')
    return filtfilt(b, a, data)

def highpass(data, cutoff=0.3, fs=FS, order=3):
    """Remove low-frequency gravity drift and keep dynamic motion."""
    b, a = butter(order, cutoff/(fs/2), btype='high')
    return filtfilt(b, a, data)

def auto_reg_berg(channel_data, name, axis, order=4):
    """Estimate autoregressive coefficients with Burg's method."""
    feats = {}
    ar_coeffs, variance, _ = arburg(channel_data, order)
    n = len(ar_coeffs) - 1

    for k in range(1, n+1):
        feats[f"{name}_{axis}_ar{k}"] = float(ar_coeffs[k])
    for k in range(n+1, order+1):
        feats[f"{name}_{axis}_ar{k}"] = 0.0
    feats[f"{name}_{axis}_ar_var"] = float(variance)
    return feats

def recurrence_rate(channel_data, name, axis, m=2, tau=1, eps=None, exclude_diag=True):
    """Compute the recurrence rate for a one-dimensional signal."""
    feats = {}

    L = len(channel_data)
    max_m = (L - 1) // tau + 1
    m_use = min(m, max_m)

    N_embed = L - (m_use - 1) * tau

    X = np.column_stack([channel_data[j * tau : j * tau + N_embed] for j in range(m_use)])
    D = squareform(pdist(X, metric='euclidean'))

    if eps is None:
        eps = 0.1 * np.std(channel_data)
    R = (D <= eps).astype(int)

    if exclude_diag:
        total = N_embed * (N_embed - 1)
        feats[f"{name}_{axis}_rr"] = (R.sum() - N_embed) / total
    else:
        total = N_embed * N_embed
        feats[f"{name}_{axis}_rr"] = R.sum() / total

    return feats

def wavelet_decomposition(channel_data, name, axis, maxlevel=5):
    """Extract wavelet packet energy, entropy, and coefficient statistics."""
    feats = {}
    wp = pywt.WaveletPacket(data=channel_data,
                         wavelet='db4',
                         mode='symmetric',
                         maxlevel=maxlevel)

    for lvl in range(1, maxlevel+1):
        nodes = wp.get_level(lvl, order='freq')
        coeffs = np.hstack([n.data for n in nodes])

        sum_abs = np.sum(np.abs(coeffs))
        feats[f"{name}_{axis}_wpd_L{lvl}_sum"] = round(sum_abs, 6)

        energy = np.sum(coeffs**2)
        feats[f"{name}_{axis}_wpd_L{lvl}_energy"] = round(energy, 6)

        psq = coeffs**2
        p = psq / (np.sum(psq) + 1e-12)
        entropy = -np.sum(p * np.log2(p + 1e-12))
        feats[f"{name}_{axis}_wpd_L{lvl}_entropy"] = round(entropy, 6)

    return feats

def permutation_entropy(channel_data, name, axis, m, tau=1, normalized=False):
    """Compute ordinal-pattern permutation entropy for a signal channel."""
    feats = {}

    N = len(channel_data)
    n_windows = N - (m - 1) * tau
    if n_windows <= 0:
        raise ValueError("Sequence is too short for the requested embedding; reduce m or tau.")

    perms = list(itertools.permutations(range(m)))
    perm_counts = Counter()

    for i in range(n_windows):
        window = channel_data[i : i + (m - 1) * tau + 1 : tau]
        pattern = tuple(np.argsort(window))
        perm_counts[pattern] += 1

    counts = np.array([perm_counts[p] for p in perms], dtype=float)
    probs = counts / counts.sum()
    probs = probs[probs > 0]

    pe = -np.sum(probs * np.log(probs))

    if normalized:
        pe /= math.log(math.factorial(m))

    feats[f"{name}_{axis}_pe"] = float(pe)

    return feats

def extract_time_domain_features(channel_data, name, axis):
    """Extract summary statistics from one time-domain signal channel."""
    feats: Dict[str,float] = {}
    seq_len = channel_data.shape[0]

    diff = np.diff(channel_data)

    # time-domain
    mean   = np.mean(channel_data)
    std    = np.std(channel_data)
    maxv   = np.max(channel_data)
    minv   = np.min(channel_data)
    med    = np.median(channel_data)

    rms        = np.sqrt(np.mean(channel_data**2))
    peak       = np.max(np.abs(channel_data))
    var        = np.var(channel_data)
    mav = np.mean(np.abs(channel_data))
    sms = np.sum(np.abs(channel_data)) / seq_len

    # slope
    t = np.arange(seq_len, dtype=channel_data.dtype)
    t_mean, channel_data_mean = t.mean(), mean
    num = ((t - t_mean) * (channel_data - channel_data_mean)).sum()
    den = ((t - t_mean)**2).sum()
    slope = num / den if den != 0 else 0.0

    zero_crossings = zero_cross_centered(channel_data)
    zc_rate = zero_crossings / (seq_len - 1)

    diff_mean  = np.mean(diff) if diff.size>0 else 0.0
    diff_rms   = np.sqrt(np.mean(diff**2)) if diff.size>0 else 0.0
    diff_std   = np.std(diff) if diff.size>0 else 0.0

    prange = maxv - minv
    total = np.sum(channel_data)
    total_abs = np.sum(np.abs(channel_data))

    iqr_v  = iqr(channel_data)
    skew_v = skew(channel_data) if std > 0 else 0.0
    kurt_v = kurtosis(channel_data, fisher=False) if std > 0 else 0.0

    feats[f"{name}_{axis}_mean"]     = mean
    feats[f"{name}_{axis}_std"]      = std
    feats[f"{name}_{axis}_max"]      = maxv
    feats[f"{name}_{axis}_min"]      = minv
    feats[f"{name}_{axis}_median"]   = med

    feats[f"{name}_{axis}_rms"]       = rms
    feats[f"{name}_{axis}_peak"]      = peak
    feats[f"{name}_{axis}_var"]       = var

    feats[f"{name}_{axis}_zc_rate"]      = zc_rate

    feats[f"{name}_{axis}_slope"]     = slope
    feats[f"{name}_{axis}_diff_mean"] = diff_mean
    feats[f"{name}_{axis}_diff_rms"]  = diff_rms
    feats[f"{name}_{axis}_diff_std"]  = diff_std

    feats[f"{name}_{axis}_range"]    = prange
    feats[f"{name}_{axis}_sum"]       = total

    feats[f"{name}_{axis}_sav"]   = total_abs
    feats[f"{name}_{axis}_mav"] = mav

    feats[f"{name}_{axis}_iqr"]      = iqr_v
    feats[f"{name}_{axis}_skew"]     = skew_v
    feats[f"{name}_{axis}_kurtosis"] = kurt_v

    feats[f"{name}_{axis}_sma"]  = sms

    return feats

def extract_frequency_domain_features(channel_data, name, axis):
    """Extract power-spectrum statistics from one signal channel."""
    feat = {}

     # 1. PSD
    # sig_detrend = detrend(channel_data)
    signal_len = channel_data.shape[0]
    nperseg = min(256, signal_len)
    nfft = max(256, 2**int(np.ceil(np.log2(nperseg))))
    freqs, psd = welch(channel_data, fs=FS, nperseg=nperseg, nfft=nfft, detrend=False)

    # 2. Band power (low/mid/high)
    bands = {"low":(0, 0.5), "mid":(0.5, 3.0), "high":(3.0, min(15.0, freqs[-1]))}
    total_power = np.trapz(psd, freqs) + 1e-8
    for band_name, (low, high) in bands.items():
        if low >= freqs[-1]:
            continue
        bp = band_power(freqs, psd, low, high)
        feat[f"{name}_{axis}_bp_fft_{band_name}"] = bp
        feat[f"{name}_{axis}_bp_fft_{band_name}_ratio"] = bp / total_power

    # 3. Dominant freq & peak
    dom_idx = np.argmax(psd[1:]) + 1
    feat[f"{name}_{axis}_fft_dom_freq"]  = freqs[dom_idx]
    feat[f"{name}_{axis}_fft_dom_power"] = psd[dom_idx]

    peaks = np.where((psd[1:-1] > psd[:-2]) & (psd[1:-1] > psd[2:]))[0] + 1
    if peaks.size>1:
        second = peaks[np.argsort(psd[peaks])[-2]]
        feat[f"{name}_{axis}_fft_2nd_peak_freq"]  = freqs[second]
        feat[f"{name}_{axis}_fft_2nd_peak_power"] = psd[second]

    # 5. Spectral Centroid
    cent = spectral_centroid(freqs, psd)
    feat[f"{name}_{axis}_fft_sp_centroid"] = cent

    # 6. Spectral Entropy & Flatness
    p_norm = psd/np.sum(psd)
    feat[f"{name}_{axis}_fft_sp_entropy"]  = entropy(p_norm)

    feat[f"{name}_{axis}_fft_skew"] = skew(psd)

    feat[f"{name}_{axis}_fft_kurtosis"] = kurtosis(psd)

    ws = np.sum(psd) + 1e-8
    weighted_freq = np.sum(freqs * psd) / ws
    feat[f"{name}_{axis}_fft_weighted_avg_freq"] = weighted_freq

    energy = np.trapz(psd**2, freqs)
    feat[f"{name}_{axis}_fft_energy"] = energy

    max_idx = int(np.argmax(psd))
    feat[f"{name}_{axis}_fft_max_idx"] = max_idx

    return feat

def extract_stft_features(channel_data, name, axis):
    """Extract summary statistics from the short-time Fourier transform."""
    signal_len = channel_data.shape[0]
    nperseg = min(128, signal_len)
    noverlap = nperseg // 2
    nfft = 2 ** int(np.ceil(np.log2(nperseg)))

    f, t, Z = stft(channel_data, fs=FS, nperseg=nperseg, noverlap=noverlap, nfft=nfft, detrend=False)
    M = np.abs(Z)  # magnitude spectrogram, shape=(len(f), len(t))
    psd = M**2

    bands = {"low":(0.0, 0.5), "mid":(0.5,3.0), "high":(3.0, min(15.0, FS/2-0.1))}
    feats = {}

    for band_name, (low, high) in bands.items():
        mask = (f >= low) & (f < high)
        frame_energy = psd[mask,:].sum(axis=0)      # shape (T,)
        feats[f'{name}_{axis}_stft_{band_name}_max']  = frame_energy.max()
        feats[f'{name}_{axis}_stft_{band_name}_mean'] = frame_energy.mean()
        feats[f'{name}_{axis}_stft_{band_name}_std']  = frame_energy.std()

    p_norm = psd / (psd.sum(axis=0, keepdims=True)+1e-8)
    ent = -np.sum(p_norm * np.log(p_norm+1e-12), axis=0)  # per-frame entropy
    feats[f"{name}_{axis}_stft_ent_mean"] = ent.mean()
    feats[f"{name}_{axis}_stft_ent_max"] = ent.max()
    feats[f"{name}_{axis}_stft_ent_std"]  = ent.std()

    # per-frame centroid
    cent = np.sum(f[:,None] * psd, axis=0) / (psd.sum(axis=0)+1e-8)
    feats[f"{name}_{axis}_stft_centroid_mean"] = cent.mean()
    feats[f"{name}_{axis}_stft_centroid_max"]  = cent.max()
    feats[f"{name}_{axis}_stft_centroid_std"]  = cent.std()

    return feats

def extract_key_acf_features(channel_data, name, axis, max_lag = 100) -> dict:
    """Extract the first local ACF peak, valley, and zero-crossing lag."""
    feats = {}
    acf = compute_acf(channel_data, max_lag)

    first_peak = None
    for k in range(1, max_lag):
        if acf[k] > acf[k-1] and acf[k] > acf[k+1]:
            first_peak = k
            break
    feats[f"{name}_{axis}_acf_first_peak_lag"] = first_peak if first_peak is not None else -1.0

    first_min = None
    for k in range(1, max_lag):
        if acf[k] < acf[k-1] and acf[k] < acf[k+1]:
            first_min = k
            break
    feats[f"{name}_{axis}_acf_first_min_lag"] = first_min if first_min is not None else -1.0

    first_zero = None
    for k in range(1, max_lag+1):
        if acf[k] <= 0 and acf[k-1] > 0:
            denominator = acf[k-1] - acf[k]
            if abs(denominator) < 1e-12:
                frac = 0.5
            else:
                frac = acf[k-1] / denominator
            first_zero = (k - 1) + frac
            break
    feats[f"{name}_{axis}_acf_first_zero_lag"] = first_zero if first_zero is not None else -1.0

    return feats

def extract_jerk_features(channel_data, name, axis):
    """Extract statistics from the first derivative of a signal channel."""
    jerk = np.diff(channel_data) / DT
    feats = {}

    if np.all(jerk == 0):
        return {
            f"{name}_{axis}_jerk_rms": 0.0,
            f"{name}_{axis}_jerk_peak": 0.0,
            f"{name}_{axis}_jerk_zc_rate": 0.0
        }

    # 2. RMS & Peak
    rms   = np.sqrt(np.mean(jerk**2) + 1e-12)
    peak  = np.max(np.abs(jerk))

    signs        = np.sign(jerk)
    nonzero_mask = signs != 0
    filtered     = signs[nonzero_mask]
    if len(filtered) < 2:
        zc_rate = 0.0
    else:
        zc      = np.sum(filtered[:-1] != filtered[1:])
        zc_rate = zc / (len(filtered)-1)

    feats[f"{name}_{axis}_jerk_rms"]     = round(rms, 4)
    feats[f"{name}_{axis}_jerk_peak"]    = round(peak, 4)
    feats[f"{name}_{axis}_jerk_zc_rate"] = round(zc_rate, 4)
    return feats

def channel_corr(data):
    """Compute within-placement correlations between accelerometer and gyroscope channels."""
    feats = {}
    channel_names = [
        "LP_acc_x", "LP_acc_y", "LP_acc_z",
        "LP_gyro_x", "LP_gyro_y", "LP_gyro_z",
        "RP_acc_x", "RP_acc_y", "RP_acc_z",
        "RP_gyro_x", "RP_gyro_y", "RP_gyro_z",
        "wrist_acc_x", "wrist_acc_y", "wrist_acc_z",
        "wrist_gyro_x", "wrist_gyro_y", "wrist_gyro_z",
        "UA_acc_x", "UA_acc_y", "UA_acc_z",
        "UA_gyro_x", "UA_gyro_y", "UA_gyro_z",
        "belt_acc_x", "belt_acc_y", "belt_acc_z",
        "belt_gyro_x", "belt_gyro_y", "belt_gyro_z"
    ]

    centered = data - data.mean(axis=1, keepdims=True)
    corr_mat = np.corrcoef(centered)

    for i in range(30):
        for j in range(i + 1, 30):
            key = f"corr_{channel_names[i]}_{channel_names[j]}"
            feats[key] = float(corr_mat[i, j])

    parts = ["LP", "RP", "wrist", "UA", "belt"]
    mag_dict = {}
    for idx, part in enumerate(parts):
        acc = np.linalg.norm(data[idx * 6 + 0: idx * 6 + 3], axis=0)
        gyro = np.linalg.norm(data[idx * 6 + 3: idx * 6 + 6], axis=0)

        mag_dict[f"{part}_acc_mag"] = acc - acc.mean()
        mag_dict[f"{part}_gyro_mag"] = gyro - gyro.mean()

    keys = list(mag_dict.keys())
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            k1, k2 = keys[i], keys[j]
            v1, v2 = mag_dict[k1], mag_dict[k2]

            if v1.std() > 1e-8 and v2.std() > 1e-8:
                corr = np.corrcoef(v1, v2)[0, 1]
            else:
                corr = 0.0

            feats[f"corr_{k1}_{k2}"] = float(corr)

    return feats

def extract_gravity_dynamic_features(acc, prefix):
    """Extract gravity-direction and dynamic-motion features from accelerometer axes."""
    feats = {}
    g_x = lowpass(acc[0], cutoff=0.1)
    g_y = lowpass(acc[1], cutoff=0.1)
    g_z = lowpass(acc[2], cutoff=0.1)
    g_mag = np.sqrt(g_x**2 + g_y**2 + g_z**2) + 1e-8
    theta_g = np.arccos(np.clip(g_z / g_mag, -1, 1))
    feats[f"{prefix}_z_grav_angle_mean"] = float(np.mean(theta_g))
    feats[f"{prefix}_z_grav_angle_std"] = float(np.std(theta_g))

    a_dyn_x = highpass(acc[0], cutoff=0.05)
    a_dyn_y = highpass(acc[1], cutoff=0.05)
    a_dyn_z = highpass(acc[2], cutoff=0.05)
    mag_dyn = np.sqrt(a_dyn_x**2 + a_dyn_y**2 + a_dyn_z**2) + 1e-8
    theta_dyn = np.arccos(np.clip(a_dyn_z / mag_dyn, -1, 1))
    feats[f"{prefix}_z_dyn_angle_mean"] = float(np.mean(theta_dyn))
    feats[f"{prefix}_z_dyn_angle_std"] = float(np.std(theta_dyn))
    feats[f"{prefix}_z_dyn_sign"] = float(np.sign(np.mean(a_dyn_z)))

    return feats

def extract_features(data):
    """
    data_: (6, seq_len) numpy.ndarray = [acc_x, acc_y, acc_z, gyr_x, gyr_y, gyr_z]
    returns: dict of rounded features
    """
    seq_len = data.shape[-1]
    assert seq_len==SEQ_LEN

    LP_acc = data[0:3, :]                    # (3, seq_len)
    LP_gyro = data[3:6, :]
    RP_acc = data[6:9, :]
    RP_gyro = data[9:12, :]
    wrist_acc = data[12:15, :]
    wrist_gyro = data[15:18, :]
    UA_acc = data[18:21, :]
    UA_gyro = data[21:24, :]
    belt_acc = data[24:27, :]
    belt_gyro = data[27:30, :]

    # prepare common constants
    sensors = {"LP_acc": LP_acc, "LP_gyro": LP_gyro,
               "RP_acc": RP_acc, "RP_gyro": RP_gyro,
               "wrist_acc": wrist_acc, "wrist_gyro": wrist_gyro,
               "UA_acc": UA_acc, "UA_gyro": UA_gyro,
               "belt_acc": belt_acc, "belt_gyro": belt_gyro}

    feats: Dict[str,float] = {}

    for name, sensor in sensors.items():
        # per-axis features

        # mag = torch.linalg.vector_norm(sensor, dim=0)       # (seq_len,)
        mag = np.linalg.norm(sensor, axis=0)        # (seq_len,)

        for idx, axis in enumerate(("x","y","z")):
            channel_data = sensor[idx]          # 1D array, length seq_len

            # time-domain
            feats.update(extract_time_domain_features(channel_data, name, axis))

            feats.update(extract_frequency_domain_features(channel_data, name, axis))
            feats.update(extract_stft_features(channel_data, name, axis))
            feats.update(extract_key_acf_features(channel_data, name, axis, min(FS, seq_len//2)))
            feats.update(extract_jerk_features(channel_data, name, axis))
            feats.update(auto_reg_berg(channel_data, name, axis, ORDER))
            feats.update(recurrence_rate(channel_data, name, axis, m=2, tau=1, eps=None, exclude_diag=True))
            feats.update(wavelet_decomposition(channel_data, name, axis, maxlevel=5))
            feats.update(permutation_entropy(channel_data, name, axis, m=3, tau=1, normalized=False))

        feats.update(extract_time_domain_features(mag, name, "mag"))
        feats.update(extract_frequency_domain_features(mag, name, "mag"))
        feats.update(extract_stft_features(mag, name, "mag"))
        feats.update(extract_key_acf_features(mag, name, "mag", min(FS, seq_len//2)))
        feats.update(extract_jerk_features(mag, name, "mag"))
        feats.update(auto_reg_berg(mag, name, "mag", ORDER))
        feats.update(recurrence_rate(mag, name, "mag", m=2, tau=1, eps=None, exclude_diag=True))
        feats.update(wavelet_decomposition(mag, name, "mag", maxlevel=5))
        feats.update(permutation_entropy(mag, name, "mag", m=3, tau=1, normalized=True))
        if "acc" in name:
            feats.update(extract_gravity_dynamic_features(sensor, prefix=name))

    # 3) cross-sensor ratio
    positions = ["LP", "RP", "wrist", "UA", "belt"]
    for p in positions:
        feats[f"{p}_intensity_ratio_acc_gyro"] = feats[f"{p}_acc_mag_rms"] / (feats[f"{p}_gyro_mag_rms"] + 1e-9)

    feats.update(channel_corr(data))

    return feats


## Extract Database Features

Compute handcrafted features for every database segment.


In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from collections import defaultdict
from autogluon.tabular import TabularPredictor

segments = all_database_segments
labels   = all_database_labels

all_database_features = []
for seg, y in zip(segments, labels):
    feats = extract_features(seg.transpose(1, 0))
    feats['activity'] = y['activity']
    feats['user_id']  = y['subject']
    all_database_features.append(feats)

all_database_features_df = pd.DataFrame(all_database_features)


## Save Feature Table

Persist the handcrafted feature table for reuse by the inference notebook.


In [ ]:
import pickle, gzip

out_path = "./dataset/shoaib/database_features.pkl.gz"
with gzip.open(out_path, "wb") as f:
    pickle.dump(all_database_features, f)


## Feature Columns

Select model input columns and exclude metadata fields.


In [ ]:
feature_cols = [c for c in all_database_features_df.columns if c not in ('activity','user_id')]


## Pairwise Modeling Scope

Enumerate activities and subjects for subject-grouped pairwise modeling.


In [ ]:
activities = sorted(all_database_features_df['activity'].unique())
subjects = sorted(all_database_features_df['user_id'].unique())


## Grouped Split Helper

Build user-grouped folds that keep both binary classes in each validation split.


In [ ]:
import numpy as np
from itertools import combinations

def safe_group_split(df, n_splits=4, seed=42, max_tries=100):
    """Build user-grouped folds that contain both binary classes in every fold."""
    users = np.sort(df['user_id'].unique())
    assert len(users) >= n_splits, "The number of users must be at least n_splits."

    rng = np.random.default_rng(seed)

    for attempt in range(max_tries):
        rng.shuffle(users)

        fold_sizes = np.full(n_splits, len(users) // n_splits)
        fold_sizes[:len(users) % n_splits] += 1

        splits = []
        idx = 0
        for fold_size in fold_sizes:
            val_users = users[idx: idx + fold_size]
            valid_mask = df['user_id'].isin(val_users).to_numpy()
            train_mask = ~valid_mask
            splits.append((
                np.where(train_mask)[0],
                np.where(valid_mask)[0]
            ))
            idx += fold_size

        ok = all(df.iloc[va]['label_bin'].nunique() == 2 for _, va in splits)
        if ok:
            return splits

        rng = np.random.default_rng(seed + attempt + 1)

    raise RuntimeError(f"Could not build class-balanced folds after {max_tries} attempts.")


## Pairwise Feature Importance

Train pairwise classifiers and aggregate feature importances across subject-grouped folds.


In [ ]:
import warnings
from sklearn.metrics import confusion_matrix

warnings.filterwarnings("ignore")

MAX_POS_W = 50
pair_map = defaultdict(dict)
# scaler = StandardScaler()

for A, B in combinations(activities, 2):

    # ------------ subset to just the two classes --------------
    sub = all_database_features_df[all_database_features_df['activity'].isin([A, B])].copy()
    sub['label_bin'] = (sub['activity'] == B).astype(int)   # B → 1,  A → 0
    print(f"**************** Training pair **************\n               {id2label[A]} vs {id2label[B]}")
    print("----------" * 4)

    # Choose n_splits from the number of available users.
    users_in_pair = sub["user_id"].unique()
    n_users_in_pair = len(users_in_pair)

    if n_users_in_pair < 2:
        print(f"Skipping pair ({id2label[A]}, {id2label[B]}) — too few users ({n_users_in_pair})")
        continue
    elif n_users_in_pair <= 4:
        N_SPLITS = 2
    elif n_users_in_pair <= 6:
        N_SPLITS = 3
    elif n_users_in_pair <= 10:
        N_SPLITS = 4
    elif n_users_in_pair <= 14:
        N_SPLITS = 5
    elif n_users_in_pair <= 18:
        N_SPLITS = 6
    else:
        N_SPLITS = min(7, n_users_in_pair)

    print(f"→ Using {N_SPLITS} folds for {n_users_in_pair} users")

    max_tries=10
    tried=0
    not_0_cols = 0

    while tried < max_tries:
        if not_0_cols > 0:
            break

        importances_accum = np.zeros(len(feature_cols))
        n_folds_used      = 0
        splits = safe_group_split(sub, n_splits=N_SPLITS, seed=42+tried)

        for fold, (tr_idx, va_idx) in enumerate(splits):
            train_df = sub.iloc[tr_idx].copy().sample(frac=1, random_state=42).reset_index(drop=True)
            valid_df = sub.iloc[va_idx].copy()

            train_users = sorted(train_df['user_id'].unique())
            valid_users = sorted(valid_df['user_id'].unique())

            # Compute the positive-class weight for this fold.
            n_pos = max((train_df.label_bin == 1).sum(), 1)
            n_neg = (train_df.label_bin == 0).sum()
            pos_w = np.clip(n_neg / n_pos, 1, MAX_POS_W)

            print(f"Fold-{fold}: Train users = {train_users}, Valid users = {valid_users}")
            print(f"Training pair with sample weight: {id2label[A]}: {1.0} vs {id2label[B]}: {pos_w} …")

            train_df['w'] = train_df['label_bin'].map({0: 1.0, 1: pos_w})
            valid_df['w'] = valid_df['label_bin'].map({0: 1.0, 1: pos_w})  # optional

            model_path = f"./dataset/shoaib/autogluon_pair_models/{A}_{B}/fold_{fold}"
            if os.path.exists(model_path):
                print("Model exists. Loading...")
                predictor = TabularPredictor.load(model_path)
            else:
                print("❌ Model not found. Training now...")
                predictor = TabularPredictor(label='label_bin',
                                         eval_metric='f1_macro',
                                         sample_weight='w',
                                         path= model_path,
                                         verbosity=0)

                predictor.fit(
                    train_data=train_df[feature_cols + ['label_bin','w']],
                    tuning_data=valid_df[feature_cols + ['label_bin','w']],
                    presets=None,
                    time_limit=10800,                   # seconds per fold
                    feature_prune_kwargs=None,
                    hyperparameters = {
                        'GBM': [{}],
                        'XGB': [{}],
                        'CAT': [{}],
                        'RF' : [{}],
                        'XT' : [{}],
                        'LR' : [{}],
                    }
                )

            # --------- permutation importance on VALID -----------
            fi = predictor.feature_importance(valid_df[feature_cols + ['label_bin', 'w']],
                                              subsample_size=None)
            fi_vec = fi['importance'].reindex(feature_cols, fill_value=0.0).values  # align

            fi_vec = np.clip(fi_vec, 0.0, None)
            fi_vec = np.round(fi_vec, 8)

            not_0_cols += (fi_vec != 0).sum()
            print("Non‑zero cols in this fold: ", (fi_vec != 0).sum(),
                  "/", len(fi_vec))

            # s = fi_vec.sum()
            # if s > 0:
            #     fi_vec = fi_vec / s

            importances_accum += fi_vec
            n_folds_used      += 1

            y_true = valid_df['label_bin'].values
            y_pred = predictor.predict(valid_df[feature_cols])

            n_pos = (y_true == 1).sum()
            n_neg = (y_true == 0).sum()
            cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

            acc_neg = cm[0, 0] / max(n_neg, 1)
            acc_pos = cm[1, 1] / max(n_pos, 1)

            print(f"Baseline F1-macro: {predictor.evaluate(valid_df[feature_cols + ['label_bin','w']], silent=True)['f1_macro']:.4f}")
            print(f"  label 0  | #samples = {n_neg:<4d} | correct = {cm[0,0]:<4d} | accuracy = {acc_neg:.3f}")
            print(f"  label 1  | #samples = {n_pos:<4d} | correct = {cm[1,1]:<4d} | accuracy = {acc_pos:.3f}")

            print("----------"*4)

        tried += 1

    # ------------- fold‑average & normalise -------------------
    # w = importances_accum / n_folds_used
    w = importances_accum
    if w.sum() > 0:
        w_norm = w / w.sum()
    else:
        w_norm = w                         # rare degenerate case

    pair_key = (id2label[A], id2label[B])
    pair_map[pair_key] = dict(zip(feature_cols, w_norm))

    print("----------"*10)


## Save Pairwise Knowledge Base

Serialize the activity-pair feature-importance map as JSON.


In [ ]:
# ------------------------------------------------------------------
# 4)  ========  Done!  pair_map ready for RAG‑LLM  ==================
# ------------------------------------------------------------------
# Optional: save to disk
import json
nested = {}
for (a, b), fmap in pair_map.items():
    nested.setdefault(str(a), {})[str(b)] = fmap

with open("./dataset/shoaib/activity_pair_f1_macro_feature_importance.json", "w") as fp:
    json.dump(nested, fp, indent=2)
print("✓ Activity‑Pair Feature Importance Map written to JSON.")


## Export Importance Table

Write a CSV summary with one row per activity pair.


In [ ]:
import pandas as pd

# ------------------------------------------------------------------
# 1) collect the complete feature list  (same list you used in training)
# ------------------------------------------------------------------
full_feats = feature_cols          # or list(pair_map[any_pair].keys())

# ------------------------------------------------------------------
# 2) re‑pack pair_map into table rows
# ------------------------------------------------------------------
rows = []
for (a, b), fmap in pair_map.items():
    row = {'act_A': a, 'act_B': b}
    # fill every feature, default 0.0
    row.update({feat: fmap.get(feat, 0.0) for feat in full_feats})
    rows.append(row)

df_pairs = pd.DataFrame(rows)

# optional: sort columns (acts first, then features)
df_pairs = df_pairs[['act_A', 'act_B', *full_feats]]

# ------------------------------------------------------------------
# 3) write to CSV
# ------------------------------------------------------------------
csv_path = "./dataset/shoaib/activity_pair_f1_macro_feature_importance.csv"
df_pairs.to_csv(csv_path, index=False)   # 6‑dec precision

print(f"✓ Saved {len(df_pairs)} rows → {csv_path}")


## Inspect Pairwise Features

Load the saved pairwise map and print selected top-ranked features for inspection.


In [ ]:
with open("./dataset/shoaib/activity_pair_f1_macro_feature_importance.json", "r") as fp:
    nested = json.load(fp)

# Flatten nested JSON to (activity_a, activity_b) -> feature-importance dict.
pair_map = {}
for a, inner in nested.items():
    for b, fmap in inner.items():
        pair_map[(a, b)] = fmap

top_n=30

for k,v in pair_map.items():
    print(f"Sum: {sum(v.values())}")

    valid_items = [(field, weight) for field, weight in v.items() if round(weight*100, 2) > 0]
    if not valid_items:
        continue
    top_feats = sorted(valid_items, key=lambda x: x[1], reverse=True)[:top_n]

    entry = f"{k[0]} vs {k[1]}:"
    for feat, w in top_feats:
        entry += f"\n   • {feat} ({round(w*100, 2)}%)"
    print(entry)

    print("------"*10)
